# Base Strategy Walkthrough

This notebook demonstrates the execution strategy framework. All logic lives in `utils/`:

| Module | What it does |
|--------|-------------|
| `config.py` | Default experiment settings (grid ranges, train/test split, etc.) |
| `preprocessing.py` | Load CSVs, detect penny vs. wide archetype, compute TWAP |
| `signals.py` | Pluggable signal functions (OI, weighted OI, etc.) + diagnostic plots |
| `strategy.py` | `BaseStrategy` class, grid search, `run_experiment()` one-liner |
| `evaluation.py` | Backtest harness, results tables, `compare_strategies()` |

**To add a new strategy:** subclass `BaseStrategy`, implement `decide()`, and call `run_experiment()`. See Section 4 for an example.

## 1. Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

from utils.config import DEFAULT_CONFIG
from utils.preprocessing import load_all_stocks, train_test_split
from utils.signals import add_horizon_columns, plot_edge_vs_imbalance
from utils.strategy import OIThresholdStrategy, TimeVaryingOIThresholdStrategy, run_experiment
from utils.evaluation import print_results, plot_results, compare_strategies

config = DEFAULT_CONFIG.copy()

data, frames, archetypes = load_all_stocks(config)

print(f"Total ticks: {len(data):,}")
for t in config['stocks']:
    n = data.loc[data['ticker'] == t, 'minute_start'].nunique()
    med = frames[t]['spread'].median()
    print(f"  {t}: {archetypes[t]:6s}  ({n} min, median spread = ${med:.4f})")

## 2. Signal Analysis

Does OI predict future price movement? Plot mean mid-price change vs. OI at multiple horizons.

In [ ]:
add_horizon_columns(data, config)
plot_edge_vs_imbalance(data, config)

## 3. Run the Base Strategy (OI Threshold)

`run_experiment()` handles everything: load data, fit parameters via grid search, backtest on held-out data.

In [ ]:
oi_results = run_experiment(OIThresholdStrategy, config, signal_fn='oi')

In [ ]:
print_results(oi_results['test_all'], strategy_name='OI Threshold')
plot_results(oi_results['test_all'], strategy_name='OI Threshold')

## 4. How to Add a New Strategy

Subclass `BaseStrategy`, implement `decide()` and `param_grid()`, then call `run_experiment()`. Here's an example using the **weighted 5-level OI** signal instead of level-1 OI:

In [ ]:
# Same strategy logic, just swap the signal to weighted 5-level OI
weighted_results = run_experiment(OIThresholdStrategy, config, signal_fn='weighted_oi')

## 5. Compare Strategies

`compare_strategies()` puts multiple experiment results side by side — mean improvement, std, win rate — with a bar chart.

In [ ]:
comparison = compare_strategies([
    ('OI (level 1)', oi_results),
    ('Weighted OI (5-level)', weighted_results),
])

## Quick Reference: Adding Your Own Strategy

```python
from utils.strategy import BaseStrategy, run_experiment
from utils.config import DEFAULT_CONFIG
import numpy as np

class MyStrategy(BaseStrategy):
    name = 'My Strategy'

    def decide(self, signal, spread, side):
        """Return tick index to execute, or None for last-tick fallback."""
        for i in range(len(signal)):
            if signal[i] > self.params['my_threshold']:
                return i
        return None

    @classmethod
    def param_grid(cls, archetype):
        return {'my_threshold': np.linspace(0.5, 0.95, 20)}

results = run_experiment(MyStrategy, DEFAULT_CONFIG, signal_fn='oi')
```

To add a new **signal**, add a function to `utils/signals.py` and register it in `SIGNAL_REGISTRY`.

## 6. Testing Teammate's Strategies (v9_final)

Using the adapter to run v9_final strategies through the same eval pipeline.

In [ ]:
time_varying_results = run_experiment(TimeVaryingOIThresholdStrategy, config, signal_fn='oi')

In [ ]:
from utils.strategies_adapter import (
    run_experiment_v9,          
    EnsembleAdapter,
    SpreadQuantileAdapter,
    OFIContrarianAdapter,
    MicropriceAdapter,
    AdaptiveDeadlineAdapter,
)

ensemble_results   = run_experiment_v9(EnsembleAdapter,        config)
spread_q_results   = run_experiment_v9(SpreadQuantileAdapter,  config)
ofi_results_v9     = run_experiment_v9(OFIContrarianAdapter,   config)
microprice_results = run_experiment_v9(MicropriceAdapter,      config)
adaptive_results   = run_experiment_v9(AdaptiveDeadlineAdapter, config)

In [ ]:
# Compare everything side by side against your baseline
compare_strategies([
    ('OI Threshold (baseline)', oi_results),       # already computed in Section 3
    ('Time-Varying OI Threshold', time_varying_results),
    ('Ensemble',                ensemble_results),
    ('SpreadQuantile',          spread_q_results),
    ('OFIContrarian',           ofi_results_v9),
    ('Microprice',              microprice_results),
    ('AdaptiveDeadline',        adaptive_results),
])
